# Lake Geneva Single-Scene Thin-Cloud Masking

This notebook demonstrates the final single-scene pipeline for a Lake Geneva example used in the paper workflow: `LC08_L1TP_195028_20240705_20240712_02_T1`. It assumes the C2L1 scene is already downloaded and the RF/XGBoost `.pkl` models have already been trained or copied into `models/general`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from lswt_cloud_masking.masking import process_scene

## Configure Paths

The default polygon is a simplified Lake Geneva outline for demonstration. For precise work, point `lake_geojson` to the full lake geometry and keep `lake_key = "geneva"`.

In [ ]:
scene = "D:/Trishna/Landsat_processing/Landsat_C2/Geneva_L1/LC08_L1TP_195028_20240705_20240712_02_T1"
lake_geojson = ROOT / "data" / "lake_geneva_simple.geojson"
lake_key = "geneva"
rf_model = ROOT / "models" / "general" / "80_10_10" / "RF_best_all_general.pkl"
xgb_model = ROOT / "models" / "general" / "80_10_10" / "XGB_best_all_general.pkl"
output_dir = ROOT / "outputs" / "geneva_20240705"

scene, lake_geojson, rf_model, xgb_model, output_dir

## Run the Masking Pipeline

The output stack has three aligned bands: operational Fmask, RF thin-cloud classes, and XGBoost thin-cloud classes.

In [ ]:
summary = process_scene(
    scene,
    lake_geojson,
    output_dir,
    lake_key=lake_key,
    rf_model_path=rf_model,
    xgb_model_path=xgb_model,
)
summary["paths"]

In [ ]:
summary["layers"]

## Quick Look

Fmask values are `0 outside/nodata`, `1 clear water`, `2 operational cloud/cirrus/shadow/dilated cloud`, `3 other`. RF/XGBoost values are `0 outside/not evaluated`, `1 thin cloud`, `2 cloud-affected`, `3 water`.

In [ ]:
import matplotlib.pyplot as plt
import rasterio

stack_path = summary["paths"]["stack"]
with rasterio.open(stack_path) as src:
    stack = src.read()
    names = [src.descriptions[i] for i in range(src.count)]

fig, axes = plt.subplots(1, stack.shape[0], figsize=(5 * stack.shape[0], 5), constrained_layout=True)
if stack.shape[0] == 1:
    axes = [axes]
for ax, arr, name in zip(axes, stack, names):
    im = ax.imshow(arr, vmin=0, vmax=3, interpolation="nearest")
    ax.set_title(name)
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.7)
plt.show()